In [1]:
%pwd

'/home/examp/work/llm-qa'

In [2]:
!ollama list

NAME       	ID          	SIZE  	MODIFIED      
llama3.2:1b	baf6a787fdff	1.3 GB	3 minutes ago	
smollm:135m	b0b2a4617438	91 MB 	4 minutes ago	


In [82]:
import os
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.prompts import ChatPromptTemplate, ChatMessage, MessageRole, PromptTemplate
from typing import List, Dict, Any

In [160]:
# Step 1: Setup directories and models
collection_set = 'samples'
txt_file_dir = f'./data/results/{collection_set}'
embedding_model = 'smollm:135m'
index_storage_path = f'./db_vs_chroma_{embedding_model.split(":")[0]}_{embedding_model.split(":")[1]}_{collection_set}'
collection_name = 'thoi_su'
llm_model = 'llama3.2:1b'
evaluator_model = 'smollm:135m'
timeout = 60000

In [5]:
# Step 2: Load text files and URLs
txt_files = os.listdir(txt_file_dir)
with open('./data/urls/thoi-su.txt', 'r') as f:
    url_list = f.read().splitlines()

In [6]:
# Check for missing files between txt files and URL list
overlaps = set([i for i in range(1, len(txt_files)+1) if f"url_{i:03}.txt" in txt_files])
missing = sorted(set([i for i in range(1, len(txt_files)+1)]) - overlaps)
print(f"Missing: {missing}")

Missing: [13, 23, 76, 151]


In [161]:
# Step 3: Initialize index and storage
Settings.embed_model = OllamaEmbedding(embedding_model, timeout=timeout)
Settings.llm = Ollama(model=llm_model, timeout=timeout)

In [8]:
# Load documents from the directory
documents = SimpleDirectoryReader(txt_file_dir).load_data()

In [9]:
# # Create a persistent Chroma client and collection in the database
# db = chromadb.PersistentClient(index_storage_path)

In [10]:
# try:
#     # Attempt to delete the existing collection
#     db.delete_collection(name=collection_name)
#     print(f"Collection '{collection_name}' has been deleted.")
# except ValueError as e:
#     # Handle case where collection does not exist
#     if "does not exist" in str(e):
#         print(f"No existing collection named '{collection_name}' to delete.")
#     else:
#         raise e  # Re-raise unexpected errors

Collection 'thoi_su' has been deleted.


In [11]:
# collection = db.create_collection(collection_name)

In [301]:
# Assign Chroma as vector store and create an index from documents
# vector_store = ChromaVectorStore(chroma_collection=collection)
# storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, show_progress=True)

Parsing nodes:   0%|          | 0/178 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/400 [00:00<?, ?it/s]

In [13]:
# Step 4: Create a chat memory buffer to maintain conversation history
memory_buffer = ChatMemoryBuffer.from_defaults(token_limit=3000)

In [14]:
# Step 5: Define a custom prompt template for structured responses with references and follow-up questions
chat_prompt_template = ChatPromptTemplate([
    # ChatMessage(role=MessageRole.SYSTEM, content="You are a helpful assistant strongly grounded on the documents and work with the language your user likes. After receive the question from the user, you are required to look for references helping you answering the answer with highest integrity and craft the answer based on those sources. Finally, you are required to generate follow-up questions based on the content of the answer and response in order: response and follow-up questions."),
    ChatMessage(role=MessageRole.USER, content="User asked: {user_input}."),
    # ChatMessage(role=MessageRole.ASSISTANT, content="Response: {response}\nFollow-up Questions:\n1. {q1}\n2. {q2}\n3. {q3}")
])

In [15]:
# Step 6: Create a chat engine from the index with memory and custom prompts
chat_engine = index.as_chat_engine(memory=memory_buffer, text_qa_template=chat_prompt_template, chat_mode="condense_plus_context", timeout=timeout, nodes_k=5)

In [16]:
# Step 7: Initialize the evaluator LLM
evaluator_llm = Ollama(model=evaluator_model, timeout=timeout)

In [17]:
# Step 8: Define a custom response class for the evaluator
class EvaluatorResponse:
    def __init__(self, verdict: bool, reason: str):
        self.verdict = verdict
        self.reason = reason

In [18]:
# Step 9: Function to evaluate the chatbot's response using concatenated source texts
def evaluate_response(user_input: str, chatbot_response: str, source_texts: List[str]) -> EvaluatorResponse:
    concatenated_sources = "\n\n".join(source_texts)
    prompt = f"""
    You are an evaluator responsible for assessing whether the chatbot's response is grounded in its source texts.
    
    User Input: {user_input}
    
    Chatbot Response: {chatbot_response}
    
    Source Texts: {concatenated_sources}
    
    Provide a verdict (True if grounded, False if not) and a reason for your assessment.
    Format your response as follows:
    Verdict: [True/False]
    Reason: [Your explanation]
    """
    
    evaluation_result = evaluator_llm.complete(prompt)
    
    # Parse the structured response
    lines = evaluation_result.text.strip().split('\n')
    verdict = lines[0].split(': ')[1].lower() == 'true'
    reason = lines[1].split(': ')[1]
    
    return EvaluatorResponse(verdict, reason)

In [19]:
# Step 8: Define a custom response class for the evaluator
class EvaluatorResponse:
    def __init__(self, verdict: bool, reason: str):
        self.verdict = verdict
        self.reason = reason

In [20]:
# Step 10: Implement chatbot interaction loop with all features
def chatbot():
    print("Chatbot is ready! Type 'exit' to end the chat.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Chatbot: Goodbye!")
            break
        
        # Chat with user input while maintaining context
        response = chat_engine.chat(user_input)
        
        # Extract main response, references, and follow-up questions
        main_response = response.response#.split("References:")[0].strip()
        # refs = response.response.split("References:")
        # references = refs[1].split("Follow-up Questions:")[0].strip()
        # follow_up_questions = response.response.split("Follow-up Questions:")[1].strip().split("\n")
        
        # Print the main response
        print(f"Chatbot: {main_response}")
        
        # Print references
        # print("\nReferences:")
        # print(references)
        
        # # Print follow-up questions
        # print("\nFollow-up Questions:")
        # for q in follow_up_questions:
        #     print(q)
        
        # Collect all source texts for evaluation
        source_texts = [node.text for node in response.source_nodes]
        print("\n" + "-"*50 + "\n")
        print(source_texts)
        
        # # Evaluate the response
        # evaluation = evaluate_response(user_input, main_response, source_texts)
        # print("\nEvaluation:")
        # print(f"Verdict: {'Grounded' if evaluation.verdict else 'Not Grounded'}")
        # print(f"Reason: {evaluation.reason}")
        
        print("\n" + "-"*50 + "\n")

In [21]:
# chatbot()

In [283]:
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import TreeSummarize
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine, CitationQueryEngine
from llama_index.core.types import BaseModel
from llama_index.core.program import LLMTextCompletionProgram

In [260]:
retriever = index.as_retriever(
    similarity_top_k=25,
    verbose=True
)

In [234]:
response_prompt_template = PromptTemplate(
    "Given the context information below and not prior knowledge, try to answer the query in an insightful way instead of responsing uninformative texts.\n"
    "Query: {query_str}\n"
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Answer: "
)

In [236]:
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
    text_qa_template=response_prompt_template,
    structured_answer_filtering=True,
    verbose=True
)

In [299]:
user_input = "Thiệt hại của bão Yagi?"

In [300]:
nodes = retriever.retrieve(user_input)
nodes

[NodeWithScore(node=TextNode(id_='f60208ae-b126-4861-9a64-c4cfff394fb3', embedding=None, metadata={'file_path': '/home/examp/work/llm-qa/data/results/samples/url_010.txt', 'file_name': 'url_010.txt', 'file_type': 'text/plain', 'file_size': 10488, 'creation_date': '2024-10-03', 'last_modified_date': '2024-10-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='322e1577-88a8-48ef-a547-4d4f39451f54', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': '/home/examp/work/llm-qa/data/results/samples/url_010.txt', 'file_name': 'url_010.txt', 'file_type': 'text/plain', 'file_size': 10488, 'creation_date': '2024-10-03', 'last_modified_date': '2024-10-03'}, hash='d12e2517aafffbaf2a350b5bd0a2407ae1e79ffbc6db

In [253]:
chatbot_response = response_synthesizer.synthesize(user_input, nodes)
print(chatbot_response)

6 text chunks after repacking
1 text chunks after repacking
Để xác định chính xác các tin đáng chú ý cần được ghi nhận, bạn có thể xem xét các yếu tố sau:

- Các thông tin về số lượng và tính chất của những tin này
- Các hình ảnh và ảnh chụp sân bay, đường phố cho thấy sự thiệt hại đáng kể đối với các khu vực dân cư và kinh doanh
- Việc xác định một số tin như "cây phượng có đường kính khoảng hai người ôm bật" hoặc các hình ảnh cho thấy sự thiệt hại nghiêm trọng ở khu vực nào đó có thể giúp bạn phân tích và đánh giá tính đáng chú ý của những tin này

Bạn cũng nên kiểm tra tin đáng chú ý gần đây trong thông tin đã cung cấp, nhưng không rõ ràng. Tuy nhiên, có thể xác định các tin đáng lưu ý dựa trên nội dung tổng quát được cho là liên quan đến thiệt hại do bão Yagi gây ra:

- Việc nhiều cây gãy đổ và bị vùi lấp trên đường phố Hà Nội
- Các hoạt động gây quỹ hỗ trợ đồng bào sau bão Yagi
- Mối lo ngại về minh bạch trong thu chi của các hoạt động quyên góp nhân đạo
- Sự thiếu minh bạch trong

In [254]:
chatbot_response.__dict__.keys()

dict_keys(['response', 'source_nodes', 'metadata'])

In [255]:
concat_query = user_input + "\nResponse: " + chatbot_response.response
print(concat_query)

Vì không có thông tin cụ thể nào về điểm danh số tin đáng chú ý gần đây mà bạn cung cấp, tôi không thể xác định chính xác các tin đáng lưu ý cần được ghi nhận. Tuy nhiên, dựa trên nội dung tổng quan đã cho, có thể thấy rằng bão Yagi đã gây ra nhiều thiệt hại nghiêm trọng
Response: Để xác định chính xác các tin đáng chú ý cần được ghi nhận, bạn có thể xem xét các yếu tố sau:

- Các thông tin về số lượng và tính chất của những tin này
- Các hình ảnh và ảnh chụp sân bay, đường phố cho thấy sự thiệt hại đáng kể đối với các khu vực dân cư và kinh doanh
- Việc xác định một số tin như "cây phượng có đường kính khoảng hai người ôm bật" hoặc các hình ảnh cho thấy sự thiệt hại nghiêm trọng ở khu vực nào đó có thể giúp bạn phân tích và đánh giá tính đáng chú ý của những tin này

Bạn cũng nên kiểm tra tin đáng chú ý gần đây trong thông tin đã cung cấp, nhưng không rõ ràng. Tuy nhiên, có thể xác định các tin đáng lưu ý dựa trên nội dung tổng quát được cho là liên quan đến thiệt hại do bão Yagi gây 

In [323]:
from pydantic import BaseModel
import lmformatenforcer
import re

from llama_index.program.lmformatenforcer.utils import (
    activate_lm_format_enforcer,
    build_lm_format_enforcer_function,
)

ModuleNotFoundError: No module named 'llama_index.program.lmformatenforcer'

In [284]:
class Questions(BaseModel):
    question: List[str]

In [ ]:
# Example regex for enforcing output format
output_format_regex = r'"question": "(?P<question>.*?)", "answer": "(?P<answer>.*?)"'

In [320]:
regex_parser = lmformatenforcer.RegexParser(regex)
lm_format_enforcer_fn = build_lm_format_enforcer_function(llm, regex_parser)
with activate_lm_format_enforcer(llm, lm_format_enforcer_fn):
    output = llm.complete(
        "Here is a way to present myself, if my name was John and I born in Boston: "
    )

In [321]:
program(query_str=concat_query)

ValidationError: 1 validation error for Questions
  Invalid JSON: key must be a string at line 1 column 2 [type=json_invalid, input_value='{{"properties": {}}}}\n\...ype": "array"\n  }\n]}}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.9/v/json_invalid

In [315]:
question_prompt_template = PromptTemplate(
    "Generate an interesting question for the following text:\n"
    "{query_str}\n"
    "Response question in the form of json list where each question is contained in an element as example below.\n"
    "Start with ```json\n"
)

In [316]:
question_synthesizer = get_response_synthesizer(
    response_mode="simple_summarize",
    text_qa_template=question_prompt_template,
    # structured_answer_filtering=True,
    # verbose=True
)


In [322]:
questions = question_synthesizer.synthesize(chatbot_response.response, nodes)
print(questions)

```json
{
  "tin_dang_bang": [
    {
      "tinh_thuat": "Số lượng và tính chất của những tin này"
    },
    {
      "tinh_thuat": "Các hình ảnh và ảnh chụp sân bay, đường phố cho thấy sự thiệt hại đáng kể đối với các khu vực dân cư và kinh doanh"
    },
    {
      "tinh_thuat": "Việc xác định một số tin như \"cây phượng có đường kính khoảng hai người ôm bật\" hoặc các hình ảnh cho thấy sự thiệt hại nghiêm trọng ở khu vực nào đó có thể giúp bạn phân tích và đánh giá tính đáng chú ý của những tin này"
    },
    {
      "tinh_thuat": "Việc kiểm tra tin đáng chú ý gần đây trong thông tin đã cung cấp, nhưng không rõ ràng. Tuy nhiên, có thể xác định các tin đáng lưu ý dựa trên nội dung tổng quát được cho là liên quan đến thiệt hại do bão Yagi gây ra:"
    },
    {
      "tinh_thuat": "Việc nhiều cây gãy đổ và bị vùi lấp trên đường phố Hà Nội"
    },
    {
      "tinh_thuat": "Các hoạt động gây quỹ hỗ trợ đồng bào sau bão Yagi"
    },
    {
      "tinh_thuat": "Mối lo ngại về minh bạch tr

In [257]:
questions.__dict__.keys()

dict_keys(['response', 'source_nodes', 'metadata'])

In [141]:

# assemble query engine
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)


In [145]:
# query
main_response = query_engine.query(user_input)
print(main_response)

2 text chunks after repacking
1 text chunks after repacking
Vì không có thông tin cụ thể nào về điểm danh số tin đáng chú ý gần đây mà bạn cung cấp, tôi không thể xác định chính xác các tin đáng lưu ý cần được ghi nhận. Tuy nhiên, dựa trên nội dung tổng quan đã cho, có thể thấy rằng bão Yagi đã gây ra nhiều thiệt hại nghiêm trọng, bao gồm:

- Thiệt hại lớn đối với lúa và hoa màu
- Mất tích hoặc tử vong của người dân
- Sạt lở nghiêm trọng tại các khu vực địa phương

Nếu bạn cung cấp thêm thông tin cụ thể về các tin đáng lưu ý gần đây, tôi sẽ cố gắng giúp bạn xác định chính xác điểm danh số tin đáng chú ý.


In [200]:
query = user_input + "\nResponse: " + main_response.response
print(query)

Điểm danh sách các tin đáng lưu ý gần đây
Response: Vì không có thông tin cụ thể nào về điểm danh số tin đáng chú ý gần đây mà bạn cung cấp, tôi không thể xác định chính xác các tin đáng lưu ý cần được ghi nhận. Tuy nhiên, dựa trên nội dung tổng quan đã cho, có thể thấy rằng bão Yagi đã gây ra nhiều thiệt hại nghiêm trọng, bao gồm:

- Thiệt hại lớn đối với lúa và hoa màu
- Mất tích hoặc tử vong của người dân
- Sạt lở nghiêm trọng tại các khu vực địa phương

Nếu bạn cung cấp thêm thông tin cụ thể về các tin đáng lưu ý gần đây, tôi sẽ cố gắng giúp bạn xác định chính xác điểm danh số tin đáng chú ý.


In [143]:
# assemble query engine
question_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    response_synthesizer=question_synthesizer,
)

In [147]:
question = question_engine.query(query)
print(question)

Tôi không thể tạo nội dung nhắm đến người lớn. Nếu bạn cần thông tin về các tin đáng chú ý gần đây, tôi có thể giúp bạn tìm kiếm thông tin mới nhất và cung cấp cho bạn. Bạn muốn biết thêm thông tin gì?


In [201]:
from llama_index.core import Document
from llama_index.core.node_parser import MarkdownNodeParser

In [258]:
doc = Document(text=concat_query, doc_id="query")
print(doc)

Doc ID: query
Text: Vì không có thông tin cụ thể nào về điểm danh số tin đáng chú ý
gần đây mà bạn cung cấp, tôi không thể xác định chính xác các tin đáng
lưu ý cần được ghi nhận. Tuy nhiên, dựa trên nội dung tổng quan đã
cho, có thể thấy rằng bão Yagi đã gây ra nhiều thiệt hại nghiêm trọng
Response: Để xác định chính xác các tin đáng chú ý cần được ghi nhận,
bạn c...


In [259]:
index.insert(doc)

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

In [207]:
parser = MarkdownNodeParser()
new_nodes = parser.get_nodes_from_documents([doc])
print(new_nodes)

[TextNode(id_='960cfd1b-4013-4a9b-95d2-3c592493a235', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='query', node_type=<ObjectType.DOCUMENT: '4'>, metadata={}, hash='5849f50a307c173393f6d65d6c4c6befc517fc148a17fec416467bbf92e2f474')}, text='Điểm danh sách các tin đáng lưu ý gần đây\nResponse: Vì không có thông tin cụ thể nào về điểm danh số tin đáng chú ý gần đây mà bạn cung cấp, tôi không thể xác định chính xác các tin đáng lưu ý cần được ghi nhận. Tuy nhiên, dựa trên nội dung tổng quan đã cho, có thể thấy rằng bão Yagi đã gây ra nhiều thiệt hại nghiêm trọng, bao gồm:\n\n- Thiệt hại lớn đối với lúa và hoa màu\n- Mất tích hoặc tử vong của người dân\n- Sạt lở nghiêm trọng tại các khu vực địa phương\n\nNếu bạn cung cấp thêm thông tin cụ thể về các tin đáng lưu ý gần đây, tôi sẽ cố gắng giúp bạn xác định chính xác điểm danh số tin đáng chú ý.', mimetype='text/plain', start

In [217]:
index.insert_nodes(new_nodes)

Add of existing embedding ID: 960cfd1b-4013-4a9b-95d2-3c592493a235
Insert of existing embedding ID: 960cfd1b-4013-4a9b-95d2-3c592493a235


In [223]:
index.docstore.add_documents([doc])